# 02 — From local interpolation to global learning

We test two progressively stronger structural hypotheses using **exactly the same 40 training states and 8 validation states**:

\[
\text{local thermodynamic proximity}\;\longrightarrow\;\text{global linear response}.
\]

The point is not only which model has the smaller error. The important question is **what scientific assumption changes**.

In [ ]:
#@title 0. Workshop setup — run once { display-mode: "form" }
# Infrastructure is intentionally hidden so workshop time stays focused on physics.

from pathlib import Path
import hashlib, importlib.util, os, shutil, subprocess, sys, time, urllib.request, zipfile

ASSET_URL = "https://github.com/Soft-Condensed-Matter/ThermoRDF-LowData-Workshop/releases/download/student-colab-v1.0-rc1/ThermoRDF-Colab-Assets.zip"
EXPECTED_ASSET_SHA256 = "2e75fd65ad39a9dec41f7b089c2aafe51a6bb14eeee049b055be8ea3953a7bea"
EXPECTED_ASSET_VERSION = "ThermoRDF Colab Assets v1.0"
WORKSHOP_ROOT = Path("/content/ThermoRDF-Workshop")
ASSET_NAME = "ThermoRDF-Colab-Assets.zip"

def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def _download(url, destination, attempts=3):
    tmp = destination.with_suffix(destination.suffix + ".part")
    if tmp.exists():
        tmp.unlink()
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            request = urllib.request.Request(
                url,
                headers={"User-Agent": "ThermoRDF-LowData-Workshop/1.0"}
            )
            with urllib.request.urlopen(request, timeout=90) as response, open(tmp, "wb") as fh:
                shutil.copyfileobj(response, fh)
            tmp.replace(destination)
            return
        except Exception as exc:
            last_error = exc
            if tmp.exists():
                tmp.unlink()
            if attempt < attempts:
                time.sleep(2 * attempt)
    raise RuntimeError(
        "Could not download the workshop assets from GitHub. "
        "Check the internet connection and rerun this cell."
    ) from last_error

def _assets_ready(root):
    required = [
        root / "COLAB_ASSET_VERSION.txt",
        root / "data/metadata/stage_02_radial_grid.csv",
        root / "data/models/stage_07_selected_b48.pt",
        root / "data/teaching/stage_02_b48_train_40.csv.gz",
        root / "data/teaching/stage_02_b48_validation_8.csv.gz",
        root / "data/teaching/stage_02_oo_reference_372.csv.gz",
        root / "src/thermordf_workshop/__init__.py",
    ]
    if not all(path.is_file() for path in required):
        return False
    return (root / "COLAB_ASSET_VERSION.txt").read_text().strip() == EXPECTED_ASSET_VERSION

# Local/instructor override used only for automated validation.
_local_root = os.environ.get("THERMORDF_WORKSHOP_ROOT", "").strip()
if _local_root:
    WORKSHOP_ROOT = Path(_local_root).resolve()
else:
    if not _assets_ready(WORKSHOP_ROOT):
        archive = Path("/content") / ASSET_NAME

        # Reuse a verified archive if this runtime already downloaded it.
        if archive.is_file() and _sha256(archive) != EXPECTED_ASSET_SHA256:
            archive.unlink()

        if not archive.is_file():
            print("Downloading workshop assets ...")
            _download(ASSET_URL, archive)

        digest = _sha256(archive)
        if digest != EXPECTED_ASSET_SHA256:
            archive.unlink(missing_ok=True)
            raise RuntimeError(
                "Workshop asset checksum mismatch. "
                "Please rerun this cell to download a clean copy."
            )

        if WORKSHOP_ROOT.exists():
            shutil.rmtree(WORKSHOP_ROOT)
        WORKSHOP_ROOT.mkdir(parents=True)

        with zipfile.ZipFile(archive) as zf:
            zf.extractall(WORKSHOP_ROOT)

        if not _assets_ready(WORKSHOP_ROOT):
            raise RuntimeError(
                "Workshop assets were downloaded but the extracted bundle is incomplete."
            )

        print("✓ Workshop assets downloaded and verified")

_required = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scikit-learn": "sklearn",
    "torch": "torch",
}
_missing = [pkg for pkg, module in _required.items() if importlib.util.find_spec(module) is None]
if _missing:
    print("Installing missing Colab packages:", ", ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

src_path = str(WORKSHOP_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from thermordf_workshop import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

torch.set_num_threads(min(2, os.cpu_count() or 1))
data = load_workshop_data(WORKSHOP_ROOT)

print(
    f"✓ Workshop ready | {len(data.train)} training + "
    f"{len(data.validation)} validation states | "
    f"{len(data.r_nm)} RDF coordinates"
)

## Hypothesis 1 — local thermodynamic smoothness

IDW asks whether nearby points in normalised \((T',P')\) space carry the most useful structural information about a target state. The distance-decay exponent is fixed at \(p=2\).

In [ ]:
idw = fit_idw(data)
idw_prediction = idw.predict_frame(data.validation)
idw_result = evaluate_predictions(data, idw_prediction)
print(f"development validation RDF RMSE: {idw_result.attrs['global_rdf_rmse']:.4f}")

### Same demanding reference state

Throughout model development we return to the same validation state, \(T=200\) K and \(P=1\) bar. Keeping the physical state fixed makes changes in reconstruction easy to interpret.

In [ ]:
plot_reference_prediction(
    data, idw_prediction, "T200_Bar0001",
    label="IDW", line_color="#1C778E", with_error=True
);

### Observe

- Where is the disagreement concentrated?
- Is the sharp first-shell structure harder than the smoother outer region?
- What does this tell you about **thermodynamic proximity as a modelling hypothesis**?

## Hypothesis 2 — one global linear structural response

Ridge replaces local interpolation with a single global relation:

\[
\widehat{\mathbf g}=\mathbf b_0 + T'\mathbf b_T + P'\mathbf b_P.
\]

All 40 training states contribute to the same response; regularisation controls the coefficient amplitudes.

In [ ]:
ridge = fit_ridge(data)
ridge_prediction = ridge.predict_frame(data.validation)
ridge_result = evaluate_predictions(data, ridge_prediction)
print(f"selected alpha: {ridge.alpha:g}")
print(f"development validation RDF RMSE: {ridge_result.attrs['global_rdf_rmse']:.4f}")

In [ ]:
plot_reference_prediction(
    data, ridge_prediction, "T200_Bar0001",
    label="Ridge", line_color="#72549A", with_error=True
);

## Compare the hypotheses

The same data now support two different structural assumptions.

In [ ]:
scores = {
    "IDW": idw_result.attrs["global_rdf_rmse"],
    "Ridge": ridge_result.attrs["global_rdf_rmse"],
}
plot_global_comparison(scores);

### Interpret

Ridge improves the development validation result relative to IDW. This supports the existence of an exploitable **global structural trend**, not only local thermodynamic smoothness.

But a linear model imposes the same form of thermodynamic sensitivity throughout the domain.

\[
\boxed{\text{Next question: is the global structural relation strictly linear?}}
\]